# Local SLM App with Ollama: Tutorial Notebook

This notebook is a learning companion to the production `.py` code.
It does **not** replace the app in `src/`; it helps you understand and run it step by step.

## What You Will Learn

1. Verify local Ollama setup and selected models.
2. Run the benchmark pipeline from the CLI.
3. Read and interpret the generated artifacts (`csv`, `json`, `md`).
4. Understand privacy vs latency vs cost tradeoffs on your own hardware.

In [ ]:
from __future__ import annotations

import csv
import json
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path('/home/ahmad/AI/Projects/local-slm-ollama-benchmark')
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'benchmark.toml'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

print(f'Project root: {PROJECT_ROOT}')
print(f'Config path:  {CONFIG_PATH}')
print(f'Artifacts:    {ARTIFACTS_DIR}')

In [ ]:
def run_cmd(cmd: str, cwd: Path | None = None) -> str:
    """Run a shell command and return stdout+stderr text."""
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True,
        cwd=str(cwd) if cwd else None,
        check=False,
    )
    output = (result.stdout + '\n' + result.stderr).strip()
    print(output)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {cmd}')
    return output

## 1) Inspect Benchmark Configuration

The benchmark is configured in `configs/benchmark.toml` and currently uses:
- `functiongemma:270m`
- `phi3.5:3.8b`
- `granite4.1:3b`

In [ ]:
print(CONFIG_PATH.read_text(encoding='utf-8'))

## 2) Verify Ollama Availability

This confirms local inference is possible and lists installed models.

In [ ]:
run_cmd('ollama --version')
run_cmd('ollama list')

## 3) (Optional) Run a Fresh Benchmark

By default this cell is safe and does nothing (`RUN_BENCHMARK = False`).
Set it to `True` when you want a new real run.

In [ ]:
RUN_BENCHMARK = False

if RUN_BENCHMARK:
    run_name = 'notebook_run_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    cmd = (
        'uv run local-slm-ollama-benchmark benchmark '
        f'--config {CONFIG_PATH} --output-dir {ARTIFACTS_DIR} --run-name {run_name}'
    )
    print(f'Running: {cmd}')
    run_cmd(cmd, cwd=PROJECT_ROOT)
    print(f'Completed run: {run_name}')
else:
    print('Skipped. Set RUN_BENCHMARK = True to execute a new benchmark run.')

## 4) Load the Latest Run Artifacts

We read the newest run directory and inspect model-level results.

In [ ]:
run_dirs = sorted([p for p in ARTIFACTS_DIR.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)
if not run_dirs:
    raise RuntimeError('No run directories found in artifacts/.')

latest_run = run_dirs[-1]
print(f'Latest run: {latest_run.name}')

model_summary_path = latest_run / 'model_summary.csv'
prompt_runs_path = latest_run / 'prompt_runs.csv'
raw_results_path = latest_run / 'raw_results.json'
report_path = latest_run / 'benchmark_report.md'

print(model_summary_path)
print(prompt_runs_path)
print(raw_results_path)
print(report_path)

In [ ]:
with model_summary_path.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))

for row in rows:
    print(
        f"{row['model']:<22} latency={float(row['avg_latency_sec']):.4f}s  "
        f"p95={float(row['p95_latency_sec']):.4f}s  tok/s={float(row['avg_tokens_per_sec']):.2f}  "
        f"quality={float(row['avg_quality_score']):.4f}  balanced={float(row['balanced_score']):.4f}"
    )

## 5) Interpret Quality-vs-Speed Tradeoff

- Lower latency and higher tok/s are better for responsiveness.
- Higher quality score indicates better rubric adherence.
- `balanced_score` combines both (quality weighted more in this project).

In [ ]:
sorted_by_quality = sorted(rows, key=lambda r: float(r['avg_quality_score']), reverse=True)
sorted_by_speed = sorted(rows, key=lambda r: float(r['avg_tokens_per_sec']), reverse=True)
sorted_by_latency = sorted(rows, key=lambda r: float(r['avg_latency_sec']))

print('Best quality: ', sorted_by_quality[0]['model'])
print('Fastest tok/s:', sorted_by_speed[0]['model'])
print('Lowest latency:', sorted_by_latency[0]['model'])

## 6) Inspect Example Responses

This helps explain *why* quality differs across models, not just *how much*.

In [ ]:
payload = json.loads(raw_results_path.read_text(encoding='utf-8'))

# Show one response per model for a selected case
target_case = 'risk_disclosure'
seen = set()
for item in payload['prompt_runs']:
    model = item['model']
    if item['case_id'] == target_case and model not in seen:
        seen.add(model)
        print('\n' + '=' * 80)
        print(f"model={model} | quality={item['quality_score']}")
        print('-' * 80)
        print(item['response'])

## 7) Privacy, Latency, Cost Checklist

Use this checklist when making production decisions:

- **Privacy**: keep prompts/outputs on-device; enforce disk encryption and access control.
- **Latency**: choose model based on p95 latency for your target UX SLA.
- **Cost**: local avoids API token billing, but include hardware + electricity + ops time.

If needed, enable cost assumptions in `configs/benchmark.toml` under `[cost]`.

## 8) Map Notebook Steps to Production Code

- CLI entrypoints: `src/local_slm_ollama_benchmark/cli.py`
- Benchmark orchestration: `src/local_slm_ollama_benchmark/benchmark.py`
- Config schema: `src/local_slm_ollama_benchmark/config.py`
- Quality rubric logic: `src/local_slm_ollama_benchmark/quality.py`

Learning in notebook, shipping with tested `.py` files keeps the project maintainable.